# Lab: BERT Fine-Tuning – Tweet Sentiment Classification

這份 Google Colab 實作會使用 BERT 將推文分類為 negative、neutral 或 positive。你會完成資料前處理、BERT 微調、validation 評估、test 評估與結果下載。

本 notebook 不需要 TWCC。請使用 Colab GPU，並只調整集中在「實驗設定」區塊的參數來提升 accuracy。

## 開始前：開啟 GPU

在 Colab 上方選單選擇 **Runtime → Change runtime type → T4 GPU**，再按 Save。依序執行每個程式區塊；第一次執行需要下載套件、資料集與預訓練模型，因此會花比較久的時間。

In [ ]:
!pip -q install --upgrade transformers==5.14.1 datasets==5.0.0
!nvidia-smi

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('請回到 Runtime 設定開啟 GPU；CPU 也能執行，但訓練會很慢。')

## 實驗設定

這一格是本 notebook 唯一需要反覆調整的地方。先使用目前設定完成一次流程，再一次只改一到兩個參數。

- `train_samples`：用多少訓練資料；增加通常能提升表現，但會增加時間。
- `batch_size`：每次更新使用多少資料；太大會造成 GPU out-of-memory。
- `epochs`：完整看過訓練集的次數；過多可能 overfit。
- `learning_rate`：每次更新的步伐；BERT 常從 `2e-5`、`3e-5` 或 `5e-5` 開始測試。
- `max_length`：每段文字的最大 token 數；較大保留更多文字，但會增加記憶體需求。

In [ ]:
model_name = 'google-bert/bert-base-uncased'
train_samples = 5000
validation_samples = 500
batch_size = 32
epochs = 3
learning_rate = 3e-5
weight_decay = 0.01
max_length = 96
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print({
    'train_samples': train_samples,
    'validation_samples': validation_samples,
    'batch_size': batch_size,
    'epochs': epochs,
    'learning_rate': learning_rate,
    'max_length': max_length,
})

### Environment Setup & Data Loading

使用 `mteb/tweet_sentiment_extraction` 資料集。原始 train split 會先以固定 seed 打亂，再切成互不重疊的 train 與 validation；test split 完全保留到最後才使用。

這樣可以避免用 test accuracy 反覆調參，造成對 test set 過度擬合。

In [ ]:
raw_train = load_dataset('mteb/tweet_sentiment_extraction', split='train').shuffle(seed=seed)
raw_test = load_dataset('mteb/tweet_sentiment_extraction', split='test')

required_samples = train_samples + validation_samples
if required_samples > len(raw_train):
    raise ValueError(f'Requested {required_samples} samples, but only {len(raw_train)} are available.')

train_dataset = raw_train.select(range(train_samples))
validation_dataset = raw_train.select(range(train_samples, required_samples))

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(validation_dataset)}')
print(f'Test samples: {len(raw_test)}')
print(train_dataset[0])

### Text Tokenization & Mini-Batching

BERT 不能直接閱讀原始字串。Tokenizer 會把文字轉成 token ID 與 attention mask。`DataCollatorWithPadding` 只把同一 batch 補到其中最長的句子，因此比把每筆資料固定補到最大長度更省 GPU 記憶體。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=max_length)

def tokenize_dataset(dataset):
    columns_to_remove = [column for column in dataset.column_names if column != 'label']
    tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=columns_to_remove)
    tokenized.set_format('torch')
    return tokenized

tokenized_train = tokenize_dataset(train_dataset)
tokenized_validation = tokenize_dataset(validation_dataset)
tokenized_test = tokenize_dataset(raw_test)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
train_loader = DataLoader(tokenized_train, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
validation_loader = DataLoader(tokenized_validation, batch_size=batch_size, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(tokenized_test, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

### Defining the BERT Classification Model

`bert-base-uncased` 已經從大量英文語料學到一般的語言知識。我們在頂端加上三類分類器，並用情緒資料微調所有權重。

分類標籤為 `0 = negative`、`1 = neutral`、`2 = positive`。

In [ ]:
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id = {label: index for index, label in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
).to(device)

print(f'Model parameters: {sum(parameter.numel() for parameter in model.parameters()):,}')

### Optimization & Loss Function

這裡使用 `AdamW` optimizer。它在 Adam 的基礎上正確處理 weight decay，是微調 Transformer 的常用選擇。learning rate scheduler 會在訓練初期 warm up，之後逐漸降低 learning rate，讓訓練更穩定。

模型會自動計算 multi-class cross-entropy loss，用來衡量預測情緒與正確標籤的差異。

In [ ]:
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
num_training_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * num_training_steps)),
    num_training_steps=num_training_steps,
)

print(f'Training steps: {num_training_steps}')

### The Training Loop (Backpropagation)

每個 batch 的學習循環如下：

1. 將 tokenized text 與 labels 移到 GPU。
2. Forward pass：BERT 產生分類 logits 與 loss。
3. `loss.backward()`：計算每個可訓練參數的梯度。
4. `optimizer.step()`：更新權重；`scheduler.step()`：調整 learning rate。
5. 每個 epoch 後在 validation split 計算 accuracy。

In [ ]:
def move_batch_to_device(batch):
    return {name: value.to(device) for name, value in batch.items()}

def evaluate_model(dataloader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for batch in dataloader:
            batch = move_batch_to_device(batch)
            outputs = model(**batch)
            batch_size_now = batch['labels'].size(0)
            total_loss += outputs.loss.item() * batch_size_now
            predictions = outputs.logits.argmax(dim=-1)
            total_correct += (predictions == batch['labels']).sum().item()
            total_examples += batch_size_now

    return total_loss / total_examples, 100.0 * total_correct / total_examples

def train_one_epoch():
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc='Training', leave=False):
        batch = move_batch_to_device(batch)
        optimizer.zero_grad()
        outputs = model(**batch)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

    return total_loss / len(train_loader)

### Model Training & Validation

以下會執行完整訓練。觀察 validation accuracy：若它持續提高，可以嘗試更多 epoch 或資料；若 training loss 下降但 validation accuracy 停滯或下降，可能發生 overfitting。

In [ ]:
history = {'train_loss': [], 'validation_loss': [], 'validation_accuracy': []}

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch()
    validation_loss, validation_accuracy = evaluate_model(validation_loader)

    history['train_loss'].append(train_loss)
    history['validation_loss'].append(validation_loss)
    history['validation_accuracy'].append(validation_accuracy)

    print(
        f'Epoch {epoch}/{epochs} | '
        f'train loss: {train_loss:.4f} | '
        f'validation loss: {validation_loss:.4f} | '
        f'validation accuracy: {validation_accuracy:.2f}%'
    )

In [ ]:
epoch_axis = range(1, epochs + 1)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epoch_axis, history['train_loss'], marker='o', label='Train loss')
plt.plot(epoch_axis, history['validation_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epoch_axis, history['validation_accuracy'], marker='o', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Validation Accuracy')
plt.ylim(0, 100)
plt.show()

### The Testing & Evaluation Loop

現在才使用完全沒有參與調參的 test split。test accuracy 是最後要記錄與比較的成績；不要根據 test accuracy 反覆調整設定。

In [ ]:
test_loss, test_accuracy = evaluate_model(test_loader)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.2f}%')

## 下載結果並繳交

下一格會建立 `bert-inf.out` 與 `bert-inf.err`，再下載到你的電腦。報告中請記錄本次的參數、validation accuracy、test accuracy 與你下一次想嘗試的方向。

In [ ]:
result_lines = [
    f'Model: {model_name}',
    f'Train samples: {train_samples}',
    f'Validation samples: {validation_samples}',
    f'Batch size: {batch_size}',
    f'Epochs: {epochs}',
    f'Learning rate: {learning_rate}',
    f'Max length: {max_length}',
    f'Best validation accuracy: {max(history["validation_accuracy"]):.4f} %',
    f'The generation accuracy is {test_accuracy:.4f} %.',
]

Path('bert-inf.out').write_text('\n'.join(result_lines) + '\n', encoding='utf-8')
Path('bert-inf.err').write_text('', encoding='utf-8')
print(Path('bert-inf.out').read_text())

In [ ]:
from google.colab import files

files.download('bert-inf.out')
files.download('bert-inf.err')

## 下一步可以嘗試什麼？

1. 先保存這次的參數與結果。
2. 一次只改一到兩個參數，例如把 `learning_rate` 改為 `2e-5`，或將 `epochs` 增加為 4。
3. 重新從「實驗設定」開始執行後面的區塊。
4. 若出現 `CUDA out of memory`，先把 `batch_size` 從 32 降為 16 或 8。

請勿更換 BERT model 或 dataset，也不要手動修改資料內容。